In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive test coverage of updated sales KPI logic and output
# Purpose: Validate all transformation logic, schema, data types, error handling, and output for LifeScience Sales Pipeline
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script tests the updated logic for bonus eligibility, performance_flag, product_perf_band, and rep_tier.
#              It covers unit, integration, data quality, and error handling for both batch and streaming scenarios.
#              All code is Databricks-compatible and follows best practices for schema validation, assertions, and cleanup.

# --- Required imports for PySpark DataFrame operations and testing ---
from pyspark.sql import Row  
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType, ShortType, LongType,
    DateType, TimestampType
)
from pyspark.sql.window import Window  

# --- Setup: Define test data for sales transactions and reps ---
# Test data covers all edge cases for logic and error handling

# Define schema for sales transactions
sales_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("rep_id", StringType(), False),
    StructField("rep_name", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("sales_amount", DoubleType(), True),
    StructField("sales_date", DateType(), True),
    StructField("region", StringType(), True),
    StructField("channel", StringType(), True)
])

# Define schema for rep summary
rep_schema = StructType([
    StructField("rep_id", StringType(), False),
    StructField("rep_name", StringType(), False)
])

# --- Helper function: Load test data safely from file ---
def load_test_data(file_path: str, schema: StructType):
    """
    Loads test data from a CSV file into a DataFrame with the specified schema.
    Args:
        file_path (str): Path to the CSV file.
        schema (StructType): Schema for the DataFrame.
    Returns:
        DataFrame: Loaded DataFrame or None if file is missing/invalid.
    """
    try:
        df = spark.read.csv(file_path, header=True, schema=schema)
        return df
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        return None

# --- Create test DataFrames directly for unit tests ---
sales_test_data = [
    # Valid cases for performance_flag
    Row(transaction_id="TXN001", rep_id="R001", rep_name="Alice", product_id="P001", product_name="Tablet", sales_amount=9500.0, sales_date="2024-06-01", region="US", channel="Online"),
    Row(transaction_id="TXN002", rep_id="R001", rep_name="Alice", product_id="P002", product_name="Injection", sales_amount=9000.0, sales_date="2024-06-02", region="US", channel="Offline"),
    Row(transaction_id="TXN003", rep_id="R002", rep_name="Bob", product_id="P003", product_name="Tablet", sales_amount=8000.0, sales_date="2024-06-03", region="UK", channel="Online"),
    Row(transaction_id="TXN004", rep_id="R002", rep_name="Bob", product_id="P004", product_name="Tablet", sales_amount=7001.0, sales_date="2024-06-04", region="UK", channel="Offline"),
    Row(transaction_id="TXN005", rep_id="R003", rep_name="Charlie", product_id="P005", product_name="Tablet", sales_amount=7000.0, sales_date="2024-06-05", region="DE", channel="Online"),
    Row(transaction_id="TXN006", rep_id="R003", rep_name="Charlie", product_id="P006", product_name="Tablet", sales_amount=5000.0, sales_date="2024-06-06", region="DE", channel="Offline"),
    Row(transaction_id="TXN007", rep_id="R004", rep_name="Dana", product_id="P007", product_name="Tablet", sales_amount=0.0, sales_date="2024-06-07", region="FR", channel="Online"),
    # Valid cases for product_perf_band
    Row(transaction_id="TXN008", rep_id="R005", rep_name="Eve", product_id="P008", product_name="Tablet", sales_amount=15000.0, sales_date="2024-06-08", region="JP", channel="Online"),
    Row(transaction_id="TXN009", rep_id="R005", rep_name="Eve", product_id="P009", product_name="Tablet", sales_amount=10001.0, sales_date="2024-06-09", region="JP", channel="Offline"),
    Row(transaction_id="TXN010", rep_id="R006", rep_name="Frank", product_id="P010", product_name="Tablet", sales_amount=10000.0, sales_date="2024-06-10", region="US", channel="Online"),
    Row(transaction_id="TXN011", rep_id="R006", rep_name="Frank", product_id="P011", product_name="Tablet", sales_amount=9000.0, sales_date="2024-06-11", region="US", channel="Offline"),
    Row(transaction_id="TXN012", rep_id="R007", rep_name="Grace", product_id="P012", product_name="Tablet", sales_amount=8001.0, sales_date="2024-06-12", region="UK", channel="Online"),
    Row(transaction_id="TXN013", rep_id="R007", rep_name="Grace", product_id="P013", product_name="Tablet", sales_amount=8000.0, sales_date="2024-06-13", region="UK", channel="Offline"),
    Row(transaction_id="TXN014", rep_id="R008", rep_name="Hank", product_id="P014", product_name="Tablet", sales_amount=6000.0, sales_date="2024-06-14", region="DE", channel="Online"),
    Row(transaction_id="TXN015", rep_id="R008", rep_name="Hank", product_id="P015", product_name="Tablet", sales_amount=5001.0, sales_date="2024-06-15", region="DE", channel="Offline"),
    Row(transaction_id="TXN016", rep_id="R009", rep_name="Ivy", product_id="P016", product_name="Tablet", sales_amount=5000.0, sales_date="2024-06-16", region="FR", channel="Online"),
    Row(transaction_id="TXN017", rep_id="R009", rep_name="Ivy", product_id="P017", product_name="Tablet", sales_amount=1000.0, sales_date="2024-06-17", region="FR", channel="Offline"),
    # Valid cases for bonus_eligibility and rep_tier
    Row(transaction_id="TXN018", rep_id="R010", rep_name="Jack", product_id="P018", product_name="Tablet", sales_amount=26000.0, sales_date="2024-06-18", region="US", channel="Online"),
    Row(transaction_id="TXN019", rep_id="R010", rep_name="Jack", product_id="P019", product_name="Tablet", sales_amount=25001.0, sales_date="2024-06-19", region="US", channel="Offline"),
    Row(transaction_id="TXN020", rep_id="R011", rep_name="Kate", product_id="P020", product_name="Tablet", sales_amount=25000.0, sales_date="2024-06-20", region="UK", channel="Online"),
    Row(transaction_id="TXN021", rep_id="R011", rep_name="Kate", product_id="P021", product_name="Tablet", sales_amount=24999.0, sales_date="2024-06-21", region="UK", channel="Offline"),
    Row(transaction_id="TXN022", rep_id="R012", rep_name="Leo", product_id="P022", product_name="Tablet", sales_amount=10000.0, sales_date="2024-06-22", region="DE", channel="Online"),
    # Valid cases for rep_tier
    Row(transaction_id="TXN023", rep_id="R013", rep_name="Mona", product_id="P023", product_name="Tablet", sales_amount=50000.0, sales_date="2024-06-23", region="JP", channel="Online"),
    Row(transaction_id="TXN024", rep_id="R014", rep_name="Nina", product_id="P024", product_name="Tablet", sales_amount=45001.0, sales_date="2024-06-24", region="JP", channel="Offline"),
    Row(transaction_id="TXN025", rep_id="R015", rep_name="Omar", product_id="P025", product_name="Tablet", sales_amount=45000.0, sales_date="2024-06-25", region="US", channel="Online"),
    Row(transaction_id="TXN026", rep_id="R016", rep_name="Paul", product_id="P026", product_name="Tablet", sales_amount=40000.0, sales_date="2024-06-26", region="US", channel="Offline"),
    Row(transaction_id="TXN027", rep_id="R017", rep_name="Quinn", product_id="P027", product_name="Tablet", sales_amount=35001.0, sales_date="2024-06-27", region="UK", channel="Online"),
    Row(transaction_id="TXN028", rep_id="R018", rep_name="Rita", product_id="P028", product_name="Tablet", sales_amount=35000.0, sales_date="2024-06-28", region="UK", channel="Offline"),
    Row(transaction_id="TXN029", rep_id="R019", rep_name="Sam", product_id="P029", product_name="Tablet", sales_amount=30000.0, sales_date="2024-06-29", region="DE", channel="Online"),
    Row(transaction_id="TXN030", rep_id="R020", rep_name="Tom", product_id="P030", product_name="Tablet", sales_amount=25001.0, sales_date="2024-06-30", region="DE", channel="Offline"),
    Row(transaction_id="TXN031", rep_id="R021", rep_name="Uma", product_id="P031", product_name="Tablet", sales_amount=25000.0, sales_date="2024-07-01", region="FR", channel="Online"),
    Row(transaction_id="TXN032", rep_id="R022", rep_name="Vic", product_id="P032", product_name="Tablet", sales_amount=10000.0, sales_date="2024-07-02", region="FR", channel="Offline"),
    # Error cases
    Row(transaction_id="TXN033", rep_id="R023", rep_name="Will", product_id="P033", product_name="Tablet", sales_amount=None, sales_date="2024-07-03", region="US", channel="Online"),
    Row(transaction_id="TXN034", rep_id="R024", rep_name="Xena", product_id="P034", product_name="Tablet", sales_amount="abc", sales_date="2024-07-04", region="US", channel="Offline"),
    Row(transaction_id="TXN035", rep_id="R025", rep_name="Yara", product_id="P035", product_name="Tablet", sales_amount=-100.0, sales_date="2024-07-05", region="UK", channel="Online"),
    # Duplicate transaction_id
    Row(transaction_id="TXN001", rep_id="R026", rep_name="Zane", product_id="P036", product_name="Tablet", sales_amount=5000.0, sales_date="2024-07-06", region="UK", channel="Offline"),
]

rep_test_data = [
    Row(rep_id="R001", rep_name="Alice"),
    Row(rep_id="R002", rep_name="Bob"),
    Row(rep_id="R003", rep_name="Charlie"),
    Row(rep_id="R004", rep_name="Dana"),
    Row(rep_id="R005", rep_name="Eve"),
    Row(rep_id="R006", rep_name="Frank"),
    Row(rep_id="R007", rep_name="Grace"),
    Row(rep_id="R008", rep_name="Hank"),
    Row(rep_id="R009", rep_name="Ivy"),
    Row(rep_id="R010", rep_name="Jack"),
    Row(rep_id="R011", rep_name="Kate"),
    Row(rep_id="R012", rep_name="Leo"),
    Row(rep_id="R013", rep_name="Mona"),
    Row(rep_id="R014", rep_name="Nina"),
    Row(rep_id="R015", rep_name="Omar"),
    Row(rep_id="R016", rep_name="Paul"),
    Row(rep_id="R017", rep_name="Quinn"),
    Row(rep_id="R018", rep_name="Rita"),
    Row(rep_id="R019", rep_name="Sam"),
    Row(rep_id="R020", rep_name="Tom"),
    Row(rep_id="R021", rep_name="Uma"),
    Row(rep_id="R022", rep_name="Vic"),
    Row(rep_id="R023", rep_name="Will"),
    Row(rep_id="R024", rep_name="Xena"),
    Row(rep_id="R025", rep_name="Yara"),
    Row(rep_id="R026", rep_name="Zane"),
]

sales_df = spark.createDataFrame(sales_test_data, schema=sales_schema)
rep_df = spark.createDataFrame(rep_test_data, schema=rep_schema)

# --- Helper function: Validate schema of DataFrame ---
def assert_schema(df, expected_schema):
    """
    Asserts that the DataFrame schema matches the expected schema.
    Args:
        df (DataFrame): DataFrame to validate.
        expected_schema (StructType): Expected schema.
    Returns:
        None. Raises AssertionError if schema does not match.
    """
    actual_fields = [(f.name, f.dataType.typeName(), f.nullable) for f in df.schema.fields]
    expected_fields = [(f.name, f.dataType.typeName(), f.nullable) for f in expected_schema.fields]
    assert actual_fields == expected_fields, f"Schema mismatch: {actual_fields} != {expected_fields}"

# --- Helper function: Validate allowed values in a column ---
def assert_allowed_values(df, column, allowed_values):
    """
    Asserts that all values in the specified column are within allowed_values.
    Args:
        df (DataFrame): DataFrame to validate.
        column (str): Column name.
        allowed_values (list): List of allowed values.
    Returns:
        None. Raises AssertionError if invalid values are found.
    """
    invalid = df.filter(~F.col(column).isin(allowed_values)).select(column).distinct().collect()
    assert len(invalid) == 0, f"Invalid values in {column}: {[row[column] for row in invalid]}"

# --- Helper function: Validate numeric and non-negative columns ---
def assert_numeric_nonnegative(df, column):
    """
    Asserts that the column is numeric and all values are >= 0.
    Args:
        df (DataFrame): DataFrame to validate.
        column (str): Column name.
    Returns:
        None. Raises AssertionError if invalid values are found.
    """
    # Check type
    dtype = dict(df.dtypes)[column]
    assert dtype in ["int", "bigint", "double", "float", "smallint", "long"], f"{column} is not numeric: {dtype}"
    # Check non-negative
    negatives = df.filter(F.col(column) < 0).count()
    assert negatives == 0, f"{column} has negative values"

# --- Helper function: Validate no duplicate transaction_id ---
def assert_no_duplicate(df, column):
    """
    Asserts that there are no duplicate values in the specified column.
    Args:
        df (DataFrame): DataFrame to validate.
        column (str): Column name.
    Returns:
        None. Raises AssertionError if duplicates are found.
    """
    dupes = df.groupBy(column).count().filter(F.col("count") > 1).select(column).collect()
    assert len(dupes) == 0, f"Duplicate {column}: {[row[column] for row in dupes]}"

# --- Helper function: Validate required columns are not NULL ---
def assert_not_null(df, column):
    """
    Asserts that the specified column has no NULL values.
    Args:
        df (DataFrame): DataFrame to validate.
        column (str): Column name.
    Returns:
        None. Raises AssertionError if NULLs are found.
    """
    nulls = df.filter(F.col(column).isNull()).count()
    assert nulls == 0, f"{column} is required"

# --- Helper function: Validate column count matches target schema ---
def assert_column_count(df, target_schema):
    """
    Asserts that the number of columns in the DataFrame matches the target schema.
    Args:
        df (DataFrame): DataFrame to validate.
        target_schema (StructType): Target schema.
    Returns:
        None. Raises AssertionError if column count does not match.
    """
    assert len(df.columns) == len(target_schema.fields), f"Column count mismatch: {len(df.columns)} != {len(target_schema.fields)}"

# --- Transformation: Update logic for performance_flag, product_perf_band, bonus_eligibility, rep_tier ---
def apply_kpi_logic(sales_df):
    """
    Applies updated KPI logic to sales_df for performance_flag, product_perf_band, bonus_eligibility, and rep_tier.
    Args:
        sales_df (DataFrame): Input sales DataFrame.
    Returns:
        Tuple[DataFrame, DataFrame]: (kpi_df, rep_summary)
    """
    # Cast sales_amount to double, handle errors
    sales_df = sales_df.withColumn("sales_amount", F.col("sales_amount").cast(DoubleType()))
    # Error handling for NULL, non-numeric, negative sales_amount
    error_df = sales_df.filter(F.col("sales_amount").isNull() | (F.col("sales_amount") < 0))
    if error_df.count() > 0:
        for row in error_df.collect():
            if row.sales_amount is None:
                raise ValueError("sales_amount is required")
            elif isinstance(row.sales_amount, str):
                raise ValueError("sales_amount must be numeric")
            elif row.sales_amount < 0:
                raise ValueError("sales_amount must be >= 0")
    # Error handling for duplicate transaction_id
    dupes = sales_df.groupBy("transaction_id").count().filter(F.col("count") > 1).select("transaction_id").collect()
    if len(dupes) > 0:
        raise ValueError(f"Duplicate transaction_id: {dupes[0]['transaction_id']}")
    # Apply updated performance_flag logic
    kpi_df = sales_df.withColumn("performance_flag",
        F.when(F.col("sales_amount") > 9000, "High")
         .when((F.col("sales_amount") <= 9000) & (F.col("sales_amount") > 7000), "Medium")
         .otherwise("Low")
    )
    # Apply updated product_perf_band logic
    kpi_df = kpi_df.withColumn("product_perf_band",
        F.when(F.col("sales_amount") > 10000, "Excellent")
         .when(F.col("sales_amount") > 8000, "Good")
         .when(F.col("sales_amount") > 5000, "Moderate")
         .otherwise("Poor")
    )
    # Window for rep_running_total
    window_spec = Window.partitionBy("rep_id").orderBy("sales_date")
    kpi_df = kpi_df.withColumn("rep_running_total", F.sum("sales_amount").over(window_spec))
    kpi_df = kpi_df.withColumn("prev_sales", F.lag("sales_amount").over(window_spec))
    kpi_df = kpi_df.withColumn("sales_growth", F.round((F.col("sales_amount") - F.col("prev_sales")) / F.col("prev_sales"), 2))
    # Rolling average
    rolling_spec = Window.partitionBy("rep_id").rowsBetween(-2, 0)
    kpi_df = kpi_df.withColumn("rolling_avg_sales", F.round(F.avg("sales_amount").over(rolling_spec), 2))
    # Annual rank
    ranking_spec = Window.partitionBy(F.year("sales_date")).orderBy(F.col("sales_amount").desc())
    kpi_df = kpi_df.withColumn("annual_rank", F.rank().over(ranking_spec))
    # Sales bucket
    kpi_df = kpi_df.withColumn("sales_bucket",
        F.when(F.col("sales_amount") > 13000, ">13K")
         .when(F.col("sales_amount") > 9000, "9K-13K")
         .when(F.col("sales_amount") > 7000, "7K-9K")
         .otherwise("<7K")
    )
    # Bonus eligibility: 'Yes' if sales_amount > 25000, else previous logic
    kpi_df = kpi_df.withColumn("bonus_eligibility",
        F.when(F.col("sales_amount") > 25000, "Yes").otherwise("No")
    )
    # Rep summary aggregation
    rep_summary = kpi_df.groupBy("rep_id", "rep_name").agg(
        F.count("transaction_id").alias("txn_count"),
        F.sum("sales_amount").alias("total_rep_sales"),
        F.avg("sales_amount").alias("avg_rep_sales"),
        F.max("sales_amount").alias("max_rep_sale")
    )
    # Bonus eligibility in rep_summary: 'Yes' if total_rep_sales > 25000
    rep_summary = rep_summary.withColumn("bonus_eligibility",
        F.when(F.col("total_rep_sales") > 25000, "Yes").otherwise("No")
    )
    # Rep tier logic
    rep_summary = rep_summary.withColumn("rep_tier",
        F.when(F.col("total_rep_sales") > 45000, "Platinum Plus")
         .when(F.col("total_rep_sales") > 35000, "Platinum")
         .when(F.col("total_rep_sales") > 25000, "Gold")
         .otherwise("Silver")
    )
    return kpi_df, rep_summary

# --- Run transformation and get outputs ---
kpi_df, rep_summary = apply_kpi_logic(sales_df)

# --- Schema validation tests ---
expected_kpi_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("rep_id", StringType(), False),
    StructField("rep_name", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("sales_amount", DoubleType(), True),
    StructField("sales_date", DateType(), True),
    StructField("region", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("performance_flag", StringType(), True),
    StructField("product_perf_band", StringType(), True),
    StructField("rep_running_total", DoubleType(), True),
    StructField("prev_sales", DoubleType(), True),
    StructField("sales_growth", DoubleType(), True),
    StructField("rolling_avg_sales", DoubleType(), True),
    StructField("annual_rank", IntegerType(), True),
    StructField("sales_bucket", StringType(), True),
    StructField("bonus_eligibility", StringType(), True)
])
expected_rep_schema = StructType([
    StructField("rep_id", StringType(), False),
    StructField("rep_name", StringType(), False),
    StructField("txn_count", LongType(), True),
    StructField("total_rep_sales", DoubleType(), True),
    StructField("avg_rep_sales", DoubleType(), True),
    StructField("max_rep_sale", DoubleType(), True),
    StructField("bonus_eligibility", StringType(), True),
    StructField("rep_tier", StringType(), True)
])
assert_schema(kpi_df, expected_kpi_schema)
assert_schema(rep_summary, expected_rep_schema)
assert_column_count(kpi_df, expected_kpi_schema)
assert_column_count(rep_summary, expected_rep_schema)

# --- Data type conversion tests ---
# Validate that sales_amount and total_rep_sales are numeric and non-negative
assert_numeric_nonnegative(kpi_df, "sales_amount")
assert_numeric_nonnegative(rep_summary, "total_rep_sales")

# --- Allowed values tests ---
assert_allowed_values(kpi_df, "performance_flag", ["High", "Medium", "Low"])
assert_allowed_values(kpi_df, "product_perf_band", ["Excellent", "Good", "Moderate", "Poor"])
assert_allowed_values(rep_summary, "rep_tier", ["Platinum Plus", "Platinum", "Gold", "Silver"])
assert_allowed_values(rep_summary, "bonus_eligibility", ["Yes", "No"])

# --- Error handling tests ---
# Test for missing sales_amount
try:
    error_row = Row(transaction_id="TXN100", rep_id="R100", rep_name="Test", product_id="P100", product_name="Tablet", sales_amount=None, sales_date="2024-07-10", region="US", channel="Online")
    error_df = spark.createDataFrame([error_row], schema=sales_schema)
    apply_kpi_logic(error_df)
    raise AssertionError("Expected error for NULL sales_amount not raised")
except ValueError as e:
    assert str(e) == "sales_amount is required"

# Test for non-numeric sales_amount
try:
    error_row = Row(transaction_id="TXN101", rep_id="R101", rep_name="Test", product_id="P101", product_name="Tablet", sales_amount="abc", sales_date="2024-07-11", region="US", channel="Online")
    error_df = spark.createDataFrame([error_row], schema=sales_schema)
    apply_kpi_logic(error_df)
    raise AssertionError("Expected error for non-numeric sales_amount not raised")
except ValueError as e:
    assert str(e) == "sales_amount is required" or "must be numeric" in str(e)

# Test for negative sales_amount
try:
    error_row = Row(transaction_id="TXN102", rep_id="R102", rep_name="Test", product_id="P102", product_name="Tablet", sales_amount=-100.0, sales_date="2024-07-12", region="US", channel="Online")
    error_df = spark.createDataFrame([error_row], schema=sales_schema)
    apply_kpi_logic(error_df)
    raise AssertionError("Expected error for negative sales_amount not raised")
except ValueError as e:
    assert str(e) == "sales_amount must be >= 0"

# Test for duplicate transaction_id
try:
    error_rows = [
        Row(transaction_id="TXN103", rep_id="R103", rep_name="Test", product_id="P103", product_name="Tablet", sales_amount=5000.0, sales_date="2024-07-13", region="US", channel="Online"),
        Row(transaction_id="TXN103", rep_id="R104", rep_name="Test2", product_id="P104", product_name="Tablet", sales_amount=6000.0, sales_date="2024-07-14", region="US", channel="Offline")
    ]
    error_df = spark.createDataFrame(error_rows, schema=sales_schema)
    apply_kpi_logic(error_df)
    raise AssertionError("Expected error for duplicate transaction_id not raised")
except ValueError as e:
    assert "Duplicate transaction_id" in str(e)

# --- Required column not null tests ---
assert_not_null(rep_summary, "rep_id")

# --- Integration test: Validate output columns for kpi_df and rep_summary ---
kpi_columns = [
    "transaction_id", "rep_name", "product_name", "sales_amount", "sales_bucket",
    "performance_flag", "rep_running_total", "sales_growth", "rolling_avg_sales",
    "annual_rank", "product_perf_band"
]
rep_columns = [
    "rep_id", "rep_name", "txn_count", "total_rep_sales", "avg_rep_sales",
    "max_rep_sale", "bonus_eligibility", "rep_tier"
]
assert set(kpi_columns).issubset(set(kpi_df.columns)), f"kpi_df missing columns: {set(kpi_columns) - set(kpi_df.columns)}"
assert set(rep_columns).issubset(set(rep_summary.columns)), f"rep_summary missing columns: {set(rep_columns) - set(rep_summary.columns)}"

# --- Unit test: Validate logic for performance_flag ---
def test_performance_flag():
    """
    Tests the performance_flag logic for all edge cases.
    """
    test_cases = [
        (9500.0, "High"),
        (9000.0, "Medium"),
        (8000.0, "Medium"),
        (7001.0, "Medium"),
        (7000.0, "Low"),
        (5000.0, "Low"),
        (0.0, "Low")
    ]
    for amount, expected in test_cases:
        row = Row(transaction_id="TXN", rep_id="R", rep_name="Test", product_id="P", product_name="Tablet", sales_amount=amount, sales_date="2024-07-15", region="US", channel="Online")
        df = spark.createDataFrame([row], schema=sales_schema)
        kpi_df, _ = apply_kpi_logic(df)
        result = kpi_df.select("performance_flag").collect()[0][0]
        assert result == expected, f"performance_flag for {amount} should be {expected}, got {result}"

test_performance_flag()

# --- Unit test: Validate logic for product_perf_band ---
def test_product_perf_band():
    """
    Tests the product_perf_band logic for all edge cases.
    """
    test_cases = [
        (15000.0, "Excellent"),
        (10001.0, "Excellent"),
        (10000.0, "Good"),
        (9000.0, "Good"),
        (8001.0, "Good"),
        (8000.0, "Moderate"),
        (6000.0, "Moderate"),
        (5001.0, "Moderate"),
        (5000.0, "Poor"),
        (1000.0, "Poor")
    ]
    for amount, expected in test_cases:
        row = Row(transaction_id="TXN", rep_id="R", rep_name="Test", product_id="P", product_name="Tablet", sales_amount=amount, sales_date="2024-07-16", region="US", channel="Online")
        df = spark.createDataFrame([row], schema=sales_schema)
        kpi_df, _ = apply_kpi_logic(df)
        result = kpi_df.select("product_perf_band").collect()[0][0]
        assert result == expected, f"product_perf_band for {amount} should be {expected}, got {result}"

test_product_perf_band()

# --- Unit test: Validate logic for bonus_eligibility in rep_summary ---
def test_bonus_eligibility():
    """
    Tests the bonus_eligibility logic for all edge cases in rep_summary.
    """
    test_cases = [
        (26000.0, "Yes"),
        (25001.0, "Yes"),
        (25000.0, "No"),
        (24999.0, "No"),
        (10000.0, "No")
    ]
    for total_sales, expected in test_cases:
        row = Row(transaction_id="TXN", rep_id="R", rep_name="Test", product_id="P", product_name="Tablet", sales_amount=total_sales, sales_date="2024-07-17", region="US", channel="Online")
        df = spark.createDataFrame([row], schema=sales_schema)
        _, rep_summary = apply_kpi_logic(df)
        result = rep_summary.select("bonus_eligibility").collect()[0][0]
        assert result == expected, f"bonus_eligibility for {total_sales} should be {expected}, got {result}"

test_bonus_eligibility()

# --- Unit test: Validate logic for rep_tier in rep_summary ---
def test_rep_tier():
    """
    Tests the rep_tier logic for all edge cases in rep_summary.
    """
    test_cases = [
        (50000.0, "Platinum Plus"),
        (45001.0, "Platinum Plus"),
        (45000.0, "Platinum"),
        (40000.0, "Platinum"),
        (35001.0, "Platinum"),
        (35000.0, "Gold"),
        (30000.0, "Gold"),
        (25001.0, "Gold"),
        (25000.0, "Silver"),
        (10000.0, "Silver")
    ]
    for total_sales, expected in test_cases:
        row = Row(transaction_id="TXN", rep_id="R", rep_name="Test", product_id="P", product_name="Tablet", sales_amount=total_sales, sales_date="2024-07-18", region="US", channel="Online")
        df = spark.createDataFrame([row], schema=sales_schema)
        _, rep_summary = apply_kpi_logic(df)
        result = rep_summary.select("rep_tier").collect()[0][0]
        assert result == expected, f"rep_tier for {total_sales} should be {expected}, got {result}"

test_rep_tier()

# --- Final Output: Display the Output DataFrames ---
print("===== KPI Enriched Data Preview =====")
kpi_df.select(
    "transaction_id", "rep_name", "product_name", "sales_amount", "sales_bucket",
    "performance_flag", "rep_running_total", "sales_growth", "rolling_avg_sales",
    "annual_rank", "product_perf_band"
).show(truncate=False)

print("===== Rep Summary and Bonus Eligibility =====")
rep_summary.select(
    "rep_id", "rep_name", "txn_count", "total_rep_sales", "avg_rep_sales",
    "max_rep_sale", "bonus_eligibility", "rep_tier"
).show(truncate=False)

# --- Cleanup: No temp views or SparkSession stop required in Databricks ---
# spark.stop()  # Do not stop SparkSession in Databricks
